# MachinaQ — AAGNet 25-Class Feature Recognition

Uses **Attributed Adjacency Graph Neural Network** (AAGNet) trained on the **MFInstSeg** dataset.  
Multi-task: semantic segmentation (25 classes) + instance segmentation + bottom-face detection.

Kernel: **StepAnalyze (Python 3.11)**

In [1]:
import os, sys
os.environ.setdefault('DGLBACKEND', 'pytorch')

from pathlib import Path
ROOT    = Path('.').resolve()
AAGNET  = ROOT / 'aagnet'
DATASET = AAGNET / 'dataset' / 'MFInstSeg'

for p in [str(ROOT), str(AAGNET)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Root :', ROOT)
print('Dataset:', DATASET)
print('Dataset exists:', DATASET.exists())

Root : E:\AiTools\AIModel_StepAnalyze
Dataset: E:\AiTools\AIModel_StepAnalyze\aagnet\dataset\MFInstSeg
Dataset exists: True


## 1. Verify environment

In [2]:
import torch
import dgl
import torchmetrics
import torch_ema

print(f'PyTorch  {torch.__version__}  CUDA: {torch.cuda.is_available()}')
print(f'DGL      {dgl.__version__}')
print(f'torchmetrics {torchmetrics.__version__}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

e:\AiTools\AIModel_StepAnalyze\conda_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch  2.10.0  CUDA: True
DGL      2.2.1
torchmetrics 1.9.0
Device: cuda


## 2. Feature classes (25)

In [3]:
FEATURE_NAMES = [
    'chamfer',               # 0
    'through_hole',          # 1
    'triangular_passage',    # 2
    'rectangular_passage',   # 3
    '6sides_passage',        # 4
    'triangular_through_slot',   # 5
    'rectangular_through_slot',  # 6
    'circular_through_slot',     # 7
    'rectangular_through_step',  # 8
    '2sides_through_step',       # 9
    'slanted_through_step',      # 10
    'Oring',                     # 11
    'blind_hole',                # 12
    'triangular_pocket',         # 13
    'rectangular_pocket',        # 14
    '6sides_pocket',             # 15
    'circular_end_pocket',       # 16
    'rectangular_blind_slot',    # 17
    'v_circular_end_blind_slot', # 18
    'h_circular_end_blind_slot', # 19
    'triangular_blind_step',     # 20
    'circular_blind_step',       # 21
    'rectangular_blind_step',    # 22
    'round',                     # 23
    'stock',                     # 24
]
print(f'{len(FEATURE_NAMES)} feature classes loaded')

25 feature classes loaded


## 3. Load dataset

In [ ]:
from dataloader.mfinstseg import MFInstSegDataset

DATASET_TYPE = 'full'   # 'full' = 25 classes | 'tiny' = 7 classes
N_CLASSES    = MFInstSegDataset.num_classes(DATASET_TYPE)
print(f'Classes: {N_CLASSES}')

train_dataset = MFInstSegDataset(
    root_dir=str(DATASET),
    split='train',
    center_and_scale=False,
    normalize=True,
    random_rotate=False,
    dataset_type=DATASET_TYPE,
    num_threads=4
)
graphs = train_dataset.graphs()  # cache — reuse for val/test

val_dataset = MFInstSegDataset(
    root_dir=str(DATASET),
    graphs=graphs,
    split='val',
    center_and_scale=False,
    normalize=True,
    dataset_type=DATASET_TYPE,
    num_threads=4
)

print(f'Train: {len(train_dataset)} samples')
print(f'Val:   {len(val_dataset)} samples')

In [ ]:
# Inspect one sample
sample = train_dataset[0]
g = sample['graph']
print(f'Graph nodes: {g.num_nodes()}  edges: {g.num_edges()}')
print(f'Node feat shape:  {g.ndata["x"].shape}')
print(f'Node grid shape:  {g.ndata["grid"].shape}')
print(f'Edge feat shape:  {g.edata["x"].shape}')
print(f'Seg labels:       {g.ndata["seg_y"].shape}  classes seen: {g.ndata["seg_y"].unique().tolist()}')
print(f'Bottom labels:    {g.ndata["bottom_y"].shape}')
print(f'Inst labels:      {sample["inst_labels"].shape}')

## 4. Dataset class distribution

In [ ]:
import matplotlib.pyplot as plt
import torch

counts = torch.zeros(N_CLASSES, dtype=torch.long)
for i in range(min(500, len(train_dataset))):
    labels = train_dataset[i]['graph'].ndata['seg_y']
    for c in labels:
        counts[c.item()] += 1

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(N_CLASSES), counts.numpy())
ax.set_xticks(range(N_CLASSES))
ax.set_xticklabels(FEATURE_NAMES, rotation=60, ha='right', fontsize=8)
ax.set_title('Face label distribution (first 500 training samples)')
ax.set_ylabel('Face count')
plt.tight_layout()
plt.show()

## 5. Build model (AAGNetSegmentor)

In [ ]:
from models.inst_segmentors import AAGNetSegmentor

CFG = dict(
    num_classes      = N_CLASSES,
    arch             = 'AAGNetGraphEncoder',  # best performer
    edge_attr_dim    = 12,
    node_attr_dim    = 10,
    edge_attr_emb    = 64,
    node_attr_emb    = 64,
    edge_grid_dim    = 0,
    node_grid_dim    = 7,
    edge_grid_emb    = 0,
    node_grid_emb    = 64,
    num_layers       = 3,
    delta            = 2,
    mlp_ratio        = 2,
    drop             = 0.25,
    drop_path        = 0.25,
    head_hidden_dim  = 64,
    conv_on_edge     = False,
    use_uv_gird      = True,
    use_edge_attr    = True,
    use_face_attr    = True,
)

model = AAGNetSegmentor(**CFG).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {trainable:,} trainable / {total:,} total')

### Optional: load pretrained weights

In [ ]:
PRETRAINED = ROOT / 'aagnet' / 'weights' / 'weight_on_MFInstseg.pth'

if PRETRAINED.exists():
    state = torch.load(str(PRETRAINED), map_location=DEVICE)
    model.load_state_dict(state)
    print('Loaded pretrained weights from', PRETRAINED.name)
else:
    print('No pretrained weights found — training from scratch')

## 6. Training

In [ ]:
import numpy as np
from tqdm.notebook import tqdm
from torch import nn
from torch_ema import ExponentialMovingAverage
from torchmetrics.classification import (
    MulticlassAccuracy, MulticlassJaccardIndex,
    BinaryAccuracy, BinaryF1Score, BinaryJaccardIndex
)

EPOCHS        = 20          # increase to 100 for full training
BATCH_SIZE    = 64
LR            = 1e-2
WEIGHT_DECAY  = 1e-2
SEG_W, INST_W, BOT_W = 1.0, 1.0, 1.0

train_loader = train_dataset.get_dataloader(batch_size=BATCH_SIZE, pin_memory=True)
val_loader   = val_dataset.get_dataloader(batch_size=BATCH_SIZE, shuffle=False,
                                           drop_last=False, pin_memory=True)

seg_loss  = nn.CrossEntropyLoss()
inst_loss = nn.BCEWithLogitsLoss()
bot_loss  = nn.BCEWithLogitsLoss()

opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=0)

iters     = len(train_loader)
ema_decay = (1.0 / 2.0) ** (1.0 / iters)
ema       = ExponentialMovingAverage(model.parameters(), decay=ema_decay)

# metrics
def make_metrics(n):
    return {
        'seg_acc':  MulticlassAccuracy(num_classes=n).to(DEVICE),
        'seg_iou':  MulticlassJaccardIndex(num_classes=n).to(DEVICE),
        'inst_acc': BinaryAccuracy().to(DEVICE),
        'inst_f1':  BinaryF1Score().to(DEVICE),
        'bot_acc':  BinaryAccuracy().to(DEVICE),
        'bot_iou':  BinaryJaccardIndex().to(DEVICE),
    }

def compute_and_reset(m):
    r = {k: v.compute().item() for k, v in m.items()}
    for v in m.values(): v.reset()
    return r

train_m = make_metrics(N_CLASSES)
val_m   = make_metrics(N_CLASSES)

history = []
best_score = 0.0
SAVE_DIR = ROOT / 'outputs'
SAVE_DIR.mkdir(exist_ok=True)

print(f'Training {EPOCHS} epochs | batch={BATCH_SIZE} | device={DEVICE}')
print(f'Train batches/epoch: {iters}')

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    train_losses = []

    for data in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [train]', leave=False):
        g          = data['graph'].to(DEVICE, non_blocking=True)
        inst_label = data['inst_labels'].to(DEVICE, non_blocking=True)
        seg_label  = g.ndata['seg_y']
        bot_label  = g.ndata['bottom_y']

        opt.zero_grad(set_to_none=True)
        seg_pred, inst_pred, bot_pred = model(g)

        loss = (SEG_W  * seg_loss(seg_pred, seg_label) +
                INST_W * inst_loss(inst_pred, inst_label) +
                BOT_W  * bot_loss(bot_pred, bot_label))
        loss.backward()
        opt.step()
        ema.update()

        train_losses.append(loss.item())
        train_m['seg_acc'].update(seg_pred, seg_label)
        train_m['seg_iou'].update(seg_pred, seg_label)
        train_m['inst_acc'].update(inst_pred, inst_label)
        train_m['inst_f1'].update(inst_pred, inst_label)
        train_m['bot_acc'].update(bot_pred, bot_label)
        train_m['bot_iou'].update(bot_pred, bot_label)

    scheduler.step()
    tr = compute_and_reset(train_m)
    tr['loss'] = float(np.mean(train_losses))

    # validation with EMA weights
    with torch.no_grad(), ema.average_parameters():
        model.eval()
        val_losses = []
        for data in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [val]', leave=False):
            g          = data['graph'].to(DEVICE)
            inst_label = data['inst_labels'].to(DEVICE)
            seg_label  = g.ndata['seg_y']
            bot_label  = g.ndata['bottom_y']

            seg_pred, inst_pred, bot_pred = model(g)
            loss = (SEG_W  * seg_loss(seg_pred, seg_label) +
                    INST_W * inst_loss(inst_pred, inst_label) +
                    BOT_W  * bot_loss(bot_pred, bot_label))
            val_losses.append(loss.item())
            val_m['seg_acc'].update(seg_pred, seg_label)
            val_m['seg_iou'].update(seg_pred, seg_label)
            val_m['inst_acc'].update(inst_pred, inst_label)
            val_m['inst_f1'].update(inst_pred, inst_label)
            val_m['bot_acc'].update(bot_pred, bot_label)
            val_m['bot_iou'].update(bot_pred, bot_label)

    vl = compute_and_reset(val_m)
    vl['loss'] = float(np.mean(val_losses))

    score = vl['seg_iou'] + vl['inst_f1'] + vl['bot_iou']
    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), str(SAVE_DIR / 'machinaq_aagnet.pth'))
        star = ' ★'
    else:
        star = ''

    row = {'epoch': epoch+1, **{f't_{k}': v for k,v in tr.items()}, **{f'v_{k}': v for k,v in vl.items()}}
    history.append(row)

    print(f"Ep {epoch+1:3d}/{EPOCHS}  "
          f"loss={tr['loss']:.4f}/{vl['loss']:.4f}  "
          f"seg_iou={tr['seg_iou']:.3f}/{vl['seg_iou']:.3f}  "
          f"inst_f1={tr['inst_f1']:.3f}/{vl['inst_f1']:.3f}  "
          f"bot_iou={tr['bot_iou']:.3f}/{vl['bot_iou']:.3f}{star}")

print(f'\nBest score: {best_score:.4f}  →  outputs/machinaq_aagnet.pth')

## 7. Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(history)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
pairs = [
    ('loss',    'Loss',           axes[0,0]),
    ('seg_acc', 'Seg Accuracy',   axes[0,1]),
    ('seg_iou', 'Seg IoU',        axes[0,2]),
    ('inst_f1', 'Inst F1',        axes[1,0]),
    ('bot_acc', 'Bottom Acc',     axes[1,1]),
    ('bot_iou', 'Bottom IoU',     axes[1,2]),
]
for key, title, ax in pairs:
    ax.plot(df['epoch'], df[f't_{key}'], label='train')
    ax.plot(df['epoch'], df[f'v_{key}'], label='val')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend()
plt.suptitle('MachinaQ — AAGNet Training (25 classes)', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Inference on a single STEP file

In [ ]:
from utils.data_utils import load_one_graph

# Load best weights
best_path = SAVE_DIR / 'machinaq_aagnet.pth'
if best_path.exists():
    model.load_state_dict(torch.load(str(best_path), map_location=DEVICE))
model.eval()

# Pick a sample from val set
sample = val_dataset[0]
g      = sample['graph'].to(DEVICE)

with torch.no_grad():
    import dgl
    bg = dgl.batch([g])
    seg_pred, inst_pred, bot_pred = model(bg)

pred_classes = seg_pred.argmax(dim=1).cpu()
true_classes = sample['graph'].ndata['seg_y'].cpu()

print(f'Faces: {pred_classes.shape[0]}')
print(f'Predicted classes: {sorted(pred_classes.unique().tolist())}')
print(f'True      classes: {sorted(true_classes.unique().tolist())}')
print(f'\nFace-level accuracy: {(pred_classes == true_classes).float().mean():.3f}')

In [ ]:
# Per-class breakdown for this part
from collections import Counter

pred_names = [FEATURE_NAMES[c] for c in pred_classes.tolist()]
true_names = [FEATURE_NAMES[c] for c in true_classes.tolist()]

print('Predicted feature distribution:')
for name, cnt in sorted(Counter(pred_names).items(), key=lambda x: -x[1]):
    print(f'  {name:<30} {cnt} faces')